# Question 2: Transition-Based Dependency Parser

## Objective

This project implements a transition-based dependency parser from scratch using the Arc-Standard transition system. The parser is trained using oracle-generated transitions from the Universal Dependencies English-EWT dataset and evaluated using Labeled Attachment Score (LAS).

In [1]:
!pip install numpy scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import urllib.request
from collections import Counter

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
base_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"

In [4]:
train_file = "en_ewt-ud-train.conllu"
dev_file = "en_ewt-ud-dev.conllu"
test_file = "en_ewt-ud-test.conllu"

In [5]:
if not os.path.exists(train_file):
    urllib.request.urlretrieve(base_url + train_file, train_file)

if not os.path.exists(dev_file):
    urllib.request.urlretrieve(base_url + dev_file, dev_file)

if not os.path.exists(test_file):
    urllib.request.urlretrieve(base_url + test_file, test_file)

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


## 1. Data Processing

The Universal Dependencies English-EWT corpus was used for this assignment. The dataset provides predefined training, development, and test splits, which were used directly without creating a new random split.

The CoNLL-U files were parsed to extract the token ID, word form, Universal POS tag (UPOS), gold HEAD, and dependency relation (DEPREL) for each token.

The dataset contains:

- Training set: 12,544 sentences
- Development set: 2,001 sentences
- Test set: 2,077 sentences

The training set was used to generate oracle-based training instances and train the transition classifier. The development set was used for the required evaluation, while the test set was used for an additional evaluation.

In [6]:
def read_conllu(filename):
    
    sentences = []
    sentence = []
    
    with open(filename, "r", encoding="utf-8") as file:
        
        for line in file:
            
            line = line.strip()
            
            # Blank line means end of sentence
            if line == "":
                
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                
                continue
            
            # Ignore comments
            if line.startswith("#"):
                continue
            
            # Split the CoNLL-U line
            columns = line.split("\t")
            
            # CoNLL-U contains 10 columns
            if len(columns) != 10:
                continue
            
            token_id = columns[0]
            
            # Ignore multi-word tokens and empty nodes
            if "-" in token_id or "." in token_id:
                continue
            
            token = {
                "id": int(columns[0]),
                "form": columns[1],
                "lemma": columns[2],
                "upos": columns[3],
                "xpos": columns[4],
                "feats": columns[5],
                "head": int(columns[6]),
                "deprel": columns[7]
            }
            
            sentence.append(token)
    
    # Add final sentence if necessary
    if sentence:
        sentences.append(sentence)
    
    return sentences

In [7]:
train_sentences = read_conllu(train_file)

dev_sentences = read_conllu(dev_file)

test_sentences = read_conllu(test_file)

In [8]:
print("Training sentences:", len(train_sentences))
print("Development sentences:", len(dev_sentences))
print("Test sentences:", len(test_sentences))

Training sentences: 12544
Development sentences: 2001
Test sentences: 2077


In [9]:
for token in train_sentences[0]:
    
    print(
        token["id"],
        token["form"],
        token["upos"],
        token["head"],
        token["deprel"]
    )

1 Al PROPN 0 root
2 - PUNCT 3 punct
3 Zaman PROPN 1 flat
4 : PUNCT 7 punct
5 American ADJ 6 amod
6 forces NOUN 7 nsubj
7 killed VERB 1 parataxis
8 Shaikh PROPN 7 obj
9 Abdullah PROPN 8 flat
10 al PROPN 8 flat
11 - PUNCT 12 punct
12 Ani PROPN 8 flat
13 , PUNCT 15 punct
14 the DET 15 det
15 preacher NOUN 8 appos
16 at ADP 18 case
17 the DET 18 det
18 mosque NOUN 15 nmod
19 in ADP 21 case
20 the DET 21 det
21 town NOUN 18 nmod
22 of ADP 23 case
23 Qaim PROPN 21 nmod
24 , PUNCT 28 punct
25 near ADP 28 case
26 the DET 28 det
27 Syrian ADJ 28 amod
28 border NOUN 21 nmod
29 . PUNCT 1 punct


## 2. Parser Configuration

The parser uses the Arc-Standard transition system. Each parser configuration consists of three components:

- **Stack:** stores the tokens currently being processed, initially containing the ROOT symbol.
- **Buffer:** contains the remaining input tokens.
- **Arcs:** stores the dependency relations created during parsing.

The parser repeatedly applies transitions to modify the stack, buffer, and dependency arcs until the sentence has been completely parsed or a stopping condition is reached.

In [10]:
class ParserConfiguration:

    def __init__(self, sentence):

        self.tokens = {
            0: {
                "id": 0,
                "form": "ROOT",
                "upos": "ROOT"
            }
        }

        for token in sentence:
            self.tokens[token["id"]] = token

        self.stack = [0]

        self.buffer = [token["id"] for token in sentence]

        self.arcs = {}

In [11]:
def is_finished(config):

    return len(config.buffer) == 0 and config.stack == [0]

## 3. Transition Operations

The Arc-Standard transition system uses three operations:

- **SHIFT:** moves the first token from the buffer to the stack.
- **LEFT-ARC(label):** makes the top stack token the head of the second stack token and removes the dependent.
- **RIGHT-ARC(label):** makes the second stack token the head of the top stack token and removes the dependent.

These transitions are used both by the oracle during training-data generation and by the trained parser during prediction.

In [12]:
def shift(config):

    if len(config.buffer) == 0:
        return False

    word = config.buffer.pop(0)

    config.stack.append(word)

    return True

In [13]:
def left_arc(config, relation):

    if len(config.stack) < 2:
        return False

    head = config.stack[-1]

    dependent = config.stack[-2]

    if dependent == 0:
        return False

    config.arcs[dependent] = (head, relation)

    config.stack.pop(-2)

    return True

In [14]:
def right_arc(config, relation):

    if len(config.stack) < 2:
        return False

    head = config.stack[-2]

    dependent = config.stack[-1]

    if dependent == 0:
        return False

    config.arcs[dependent] = (head, relation)

    config.stack.pop()

    return True

In [15]:
sentence = train_sentences[0]

config = ParserConfiguration(sentence)

print("Stack:", config.stack)

print("Buffer:", config.buffer)

print("Arcs:", config.arcs)

Stack: [0]
Buffer: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {}


In [16]:
shift(config)

print("After SHIFT:")

print("Stack:", config.stack)

print("Buffer:", config.buffer)

print("Arcs:", config.arcs)

After SHIFT:
Stack: [0, 1]
Buffer: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {}


In [17]:
# Test LEFT-ARC

config = ParserConfiguration(train_sentences[0])

shift(config)
shift(config)

print("Before LEFT-ARC:")
print("Stack:", config.stack)
print("Buffer:", config.buffer)
print("Arcs:", config.arcs)

left_arc(config, "test")

print("\nAfter LEFT-ARC:")
print("Stack:", config.stack)
print("Buffer:", config.buffer)
print("Arcs:", config.arcs)

Before LEFT-ARC:
Stack: [0, 1, 2]
Buffer: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {}

After LEFT-ARC:
Stack: [0, 2]
Buffer: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {1: (2, 'test')}


In [18]:
# Test RIGHT-ARC

config = ParserConfiguration(train_sentences[0])

shift(config)
shift(config)

print("Before RIGHT-ARC:")
print("Stack:", config.stack)
print("Buffer:", config.buffer)
print("Arcs:", config.arcs)

right_arc(config, "test")

print("\nAfter RIGHT-ARC:")
print("Stack:", config.stack)
print("Buffer:", config.buffer)
print("Arcs:", config.arcs)

Before RIGHT-ARC:
Stack: [0, 1, 2]
Buffer: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {}

After RIGHT-ARC:
Stack: [0, 1]
Buffer: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Arcs: {2: (1, 'test')}


In [19]:
# Get gold dependency information

sentence = train_sentences[0]

gold_heads = {
    token["id"]: token["head"]
    for token in sentence
}

gold_deprels = {
    token["id"]: token["deprel"]
    for token in sentence
}

print("Gold Heads:")
print(gold_heads)

print("\nGold Dependency Relations:")
print(gold_deprels)

Gold Heads:
{1: 0, 2: 3, 3: 1, 4: 7, 5: 6, 6: 7, 7: 1, 8: 7, 9: 8, 10: 8, 11: 12, 12: 8, 13: 15, 14: 15, 15: 8, 16: 18, 17: 18, 18: 15, 19: 21, 20: 21, 21: 18, 22: 23, 23: 21, 24: 28, 25: 28, 26: 28, 27: 28, 28: 21, 29: 1}

Gold Dependency Relations:
{1: 'root', 2: 'punct', 3: 'flat', 4: 'punct', 5: 'amod', 6: 'nsubj', 7: 'parataxis', 8: 'obj', 9: 'flat', 10: 'flat', 11: 'punct', 12: 'flat', 13: 'punct', 14: 'det', 15: 'appos', 16: 'case', 17: 'det', 18: 'nmod', 19: 'case', 20: 'det', 21: 'nmod', 22: 'case', 23: 'nmod', 24: 'punct', 25: 'case', 26: 'det', 27: 'amod', 28: 'nmod', 29: 'punct'}


## 4. Oracle Simulation

The oracle determines the correct transition at each parser configuration using the gold dependency tree. It selects among SHIFT, LEFT-ARC(label), and RIGHT-ARC(label).

An arc transition is selected when the corresponding gold head relationship is satisfied and the dependent has no remaining unattached children. This ensures that a dependent is removed from the stack only after all of its required dependents have been processed.

The oracle-generated transition sequences are used to create supervised training instances for the classifier.

In [20]:
def has_unattached_children(word_id, config, gold_heads):

    for dependent, head in gold_heads.items():

        if head == word_id:

            if dependent not in config.arcs:
                return True

    return False

In [21]:
def oracle(config, gold_heads, gold_deprels):

    # Check if there are at least two words on the stack
    if len(config.stack) >= 2:

        s0 = config.stack[-2]
        s1 = config.stack[-1]

        # LEFT-ARC
        # The top of the stack becomes the head
        # of the second item on the stack
        if (
            s0 != 0
            and gold_heads[s0] == s1
            and not has_unattached_children(
                s0,
                config,
                gold_heads
            )
        ):
            return "LEFT-ARC:" + gold_deprels[s0]

        # RIGHT-ARC
        # The second item on the stack becomes
        # the head of the top item
        if (
            gold_heads[s1] == s0
            and not has_unattached_children(
                s1,
                config,
                gold_heads
            )
        ):
            return "RIGHT-ARC:" + gold_deprels[s1]

    # If no ARC is possible, perform SHIFT
    if len(config.buffer) > 0:
        return "SHIFT"

    return None

In [22]:
config = ParserConfiguration(sentence)

print("Initial Stack:")
print(config.stack)

print("\nInitial Buffer:")
print(config.buffer)

print("\nOracle Decision:")
print(
    oracle(
        config,
        gold_heads,
        gold_deprels
    )
)

Initial Stack:
[0]

Initial Buffer:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

Oracle Decision:
SHIFT


In [23]:
config = ParserConfiguration(sentence)

step = 1

while not is_finished(config):

    transition = oracle(
        config,
        gold_heads,
        gold_deprels
    )

    print("Step", step, ":", transition)

    if transition is None:
        print("Oracle could not find a transition.")
        break

    if transition == "SHIFT":

        shift(config)

    elif transition.startswith("LEFT-ARC:"):

        relation = transition.split(":", 1)[1]

        left_arc(
            config,
            relation
        )

    elif transition.startswith("RIGHT-ARC:"):

        relation = transition.split(":", 1)[1]

        right_arc(
            config,
            relation
        )

    step += 1

Step 1 : SHIFT
Step 2 : SHIFT
Step 3 : SHIFT
Step 4 : LEFT-ARC:punct
Step 5 : RIGHT-ARC:flat
Step 6 : SHIFT
Step 7 : SHIFT
Step 8 : SHIFT
Step 9 : LEFT-ARC:amod
Step 10 : SHIFT
Step 11 : LEFT-ARC:nsubj
Step 12 : LEFT-ARC:punct
Step 13 : SHIFT
Step 14 : SHIFT
Step 15 : RIGHT-ARC:flat
Step 16 : SHIFT
Step 17 : RIGHT-ARC:flat
Step 18 : SHIFT
Step 19 : SHIFT
Step 20 : LEFT-ARC:punct
Step 21 : RIGHT-ARC:flat
Step 22 : SHIFT
Step 23 : SHIFT
Step 24 : SHIFT
Step 25 : LEFT-ARC:det
Step 26 : LEFT-ARC:punct
Step 27 : SHIFT
Step 28 : SHIFT
Step 29 : SHIFT
Step 30 : LEFT-ARC:det
Step 31 : LEFT-ARC:case
Step 32 : SHIFT
Step 33 : SHIFT
Step 34 : SHIFT
Step 35 : LEFT-ARC:det
Step 36 : LEFT-ARC:case
Step 37 : SHIFT
Step 38 : SHIFT
Step 39 : LEFT-ARC:case
Step 40 : RIGHT-ARC:nmod
Step 41 : SHIFT
Step 42 : SHIFT
Step 43 : SHIFT
Step 44 : SHIFT
Step 45 : SHIFT
Step 46 : LEFT-ARC:amod
Step 47 : LEFT-ARC:det
Step 48 : LEFT-ARC:case
Step 49 : LEFT-ARC:punct
Step 50 : RIGHT-ARC:nmod
Step 51 : RIGHT-ARC:nmod


## 5. Feature Extraction

For each parser configuration, four POS-based features are extracted to represent the current parsing state:

1. POS tag of the top word on the stack.
2. POS tag of the second word on the stack.
3. POS tag of the first word in the buffer.
4. POS tag of the second word in the buffer.

These categorical features are converted into numerical representations using a dictionary vectorizer before being provided to the classifier.

In [24]:
def get_pos(config, token_id):

    if token_id is None:
        return "NULL"

    return config.tokens[token_id]["upos"]

In [25]:
def extract_features(config):

    # Top word on the stack
    if len(config.stack) >= 1:
        s1 = config.stack[-1]
    else:
        s1 = None

    # Second word on the stack
    if len(config.stack) >= 2:
        s2 = config.stack[-2]
    else:
        s2 = None

    # First word in the buffer
    if len(config.buffer) >= 1:
        b1 = config.buffer[0]
    else:
        b1 = None

    # Second word in the buffer
    if len(config.buffer) >= 2:
        b2 = config.buffer[1]
    else:
        b2 = None

    features = {
        "stack_top_pos": get_pos(config, s1),
        "stack_second_pos": get_pos(config, s2),
        "buffer_first_pos": get_pos(config, b1),
        "buffer_second_pos": get_pos(config, b2)
    }

    return features

In [26]:
config = ParserConfiguration(train_sentences[0])

features = extract_features(config)

print("Extracted Features:")
print(features)

Extracted Features:
{'stack_top_pos': 'ROOT', 'stack_second_pos': 'NULL', 'buffer_first_pos': 'PROPN', 'buffer_second_pos': 'PUNCT'}


In [27]:
def generate_training_data(sentences):

    X = []
    y = []

    failed_sentences = 0

    for sentence in sentences:

        # Gold heads
        gold_heads = {
            token["id"]: token["head"]
            for token in sentence
        }

        # Gold dependency relations
        gold_deprels = {
            token["id"]: token["deprel"]
            for token in sentence
        }

        # Initial parser configuration
        config = ParserConfiguration(sentence)

        steps = 0
        max_steps = 4 * len(sentence) + 10

        while not is_finished(config):

            steps += 1

            # Safety check
            if steps > max_steps:
                failed_sentences += 1
                break

            # Get correct transition from oracle
            transition = oracle(
                config,
                gold_heads,
                gold_deprels
            )

            if transition is None:
                failed_sentences += 1
                break

            # Extract features before applying transition
            features = extract_features(config)

            # Store training example
            X.append(features)
            y.append(transition)

            # Apply the oracle transition
            if transition == "SHIFT":

                shift(config)

            elif transition.startswith("LEFT-ARC:"):

                relation = transition.split(":", 1)[1]

                left_arc(
                    config,
                    relation
                )

            elif transition.startswith("RIGHT-ARC:"):

                relation = transition.split(":", 1)[1]

                right_arc(
                    config,
                    relation
                )

    print("Training examples:", len(X))
    print("Failed sentences:", failed_sentences)

    return X, y

In [28]:
X_small, y_small = generate_training_data(
    train_sentences[:100]
)

Training examples: 4584
Failed sentences: 6


In [29]:
print("Number of training examples:", len(y_small))

print("\nFirst 30 transition labels:")

for label in y_small[:30]:
    print(label)

Number of training examples: 4584

First 30 transition labels:
SHIFT
SHIFT
SHIFT
LEFT-ARC:punct
RIGHT-ARC:flat
SHIFT
SHIFT
SHIFT
LEFT-ARC:amod
SHIFT
LEFT-ARC:nsubj
LEFT-ARC:punct
SHIFT
SHIFT
RIGHT-ARC:flat
SHIFT
RIGHT-ARC:flat
SHIFT
SHIFT
LEFT-ARC:punct
RIGHT-ARC:flat
SHIFT
SHIFT
SHIFT
LEFT-ARC:det
LEFT-ARC:punct
SHIFT
SHIFT
SHIFT
LEFT-ARC:det


In [30]:
from collections import Counter

transition_counts = Counter()

for label in y_small:

    if label == "SHIFT":
        transition_counts["SHIFT"] += 1

    elif label.startswith("LEFT-ARC:"):
        transition_counts["LEFT-ARC"] += 1

    elif label.startswith("RIGHT-ARC:"):
        transition_counts["RIGHT-ARC"] += 1

print("Transition counts:")
print(transition_counts)

Transition counts:
Counter({'SHIFT': 2311, 'LEFT-ARC': 1398, 'RIGHT-ARC': 875})


In [31]:
X_train, y_train = generate_training_data(
    train_sentences
)

Training examples: 407165
Failed sentences: 287


## 6. Model Training

The oracle-generated configurations are used as supervised training instances. Each instance consists of the four extracted POS features and the corresponding oracle transition as the target label.

The categorical POS features are converted into numerical form using `DictVectorizer`. A Logistic Regression classifier is then trained to predict the next parser transition from the current configuration.

The complete training set contains 407,165 generated training instances.

In [33]:
vectorizer = DictVectorizer()

X_train_vectorized = vectorizer.fit_transform(X_train)

print("Training examples:", len(X_train))
print("Feature matrix shape:", X_train_vectorized.shape)

Training examples: 407165
Feature matrix shape: (407165, 73)


In [34]:
classifier = LogisticRegression(
    max_iter=200,
    solver="lbfgs"
)

print("Classifier created successfully.")

Classifier created successfully.


In [35]:
classifier.fit(
    X_train_vectorized,
    y_train
)

print("Model training completed successfully.")

Model training completed successfully.


## 7. Parser Implementation

After training, the classifier is used to predict the next transition for a given parser configuration. The parser extracts the four POS-based features, converts them using the fitted vectorizer, and passes them to the trained Logistic Regression classifier.

The predicted transition is applied to update the stack, buffer, and dependency arcs. This process is repeated until the sentence is completely parsed or a stopping condition is reached.

In [36]:
def predict_transition(config):

    # Extract the four POS features
    features = extract_features(config)

    # Convert features into the same format
    # used during model training
    X = vectorizer.transform([features])

    # Predict the next transition
    prediction = classifier.predict(X)[0]

    return prediction

In [48]:
def parse_sentence(sentence):

    # Create the initial parser configuration
    config = ParserConfiguration(sentence)

    steps = 0
    max_steps = 4 * len(sentence) + 20

    while not is_finished(config):

        steps += 1

        # Safety check to prevent infinite loops
        if steps > max_steps:
            print("Maximum number of steps reached.")
            break

        # Predict the next transition using the trained classifier
        transition = predict_transition(config)

        # SHIFT is valid only when the buffer is not empty
        if transition == "SHIFT":

            if len(config.buffer) > 0:
                shift(config)
            else:
                break

        # LEFT-ARC
        elif transition.startswith("LEFT-ARC:"):

            relation = transition.split(":", 1)[1]

            # LEFT-ARC requires at least two stack items
            # and the second item cannot be ROOT
            if len(config.stack) >= 2 and config.stack[-2] != 0:
                left_arc(config, relation)
            else:
                break

        # RIGHT-ARC
        elif transition.startswith("RIGHT-ARC:"):

            relation = transition.split(":", 1)[1]

            # RIGHT-ARC requires at least two stack items
            # and the dependent cannot be ROOT
            if len(config.stack) >= 2 and config.stack[-1] != 0:
                right_arc(config, relation)
            else:
                break

        else:
            # Unknown transition
            break

    return config

In [51]:
sentence = test_sentences[0]

parsed = parse_sentence(sentence)

print("Parsing completed successfully.")
print("Number of predicted arcs:", len(parsed.arcs))

Parsing completed successfully.
Number of predicted arcs: 7


In [39]:
print("Sentence:")
print(" ".join(token["form"] for token in sentence))

print("\nPredicted Dependencies:")

for token in sentence:

    token_id = token["id"]

    word = token["form"]

    if token_id in parsed.arcs:

        head_id, relation = parsed.arcs[token_id]

        head_word = parsed.tokens[head_id]["form"]

        print(
            f"{word} -> {head_word} ({relation})"
        )

Sentence:
What if Google Morphed Into GoogleOS ?

Predicted Dependencies:
What -> Morphed (nsubj)
if -> Morphed (mark)
Google -> Morphed (nsubj)
Morphed -> ROOT (root)
Into -> GoogleOS (case)
GoogleOS -> Morphed (obl)
? -> Morphed (punct)


## 8. Evaluation and Results

The parser is evaluated using Labeled Attachment Score (LAS). LAS measures the percentage of words for which both the predicted head and the predicted dependency relation are correct.

As specified in the assignment, the trained parser was evaluated on the Universal Dependencies English-EWT development set (`en_ewt-ud-dev.conllu`).

### Results

- Development LAS: **56.71%**

An additional evaluation was performed on the test set, resulting in a Test LAS of **57.17%**.

The required development-set LAS is **56.71%**.

In [40]:
def calculate_las(sentence, parsed):

    correct = 0
    total = len(sentence)

    for token in sentence:

        token_id = token["id"]

        gold_head = token["head"]
        gold_relation = token["deprel"]

        # Check whether a prediction exists
        if token_id not in parsed.arcs:
            continue

        predicted_head, predicted_relation = parsed.arcs[token_id]

        # Both head and dependency relation must be correct
        if (
            predicted_head == gold_head
            and predicted_relation == gold_relation
        ):
            correct += 1

    return correct / total

In [41]:
las = calculate_las(
    sentence,
    parsed
)

print(f"LAS for this sentence: {las * 100:.2f}%")

LAS for this sentence: 71.43%


In [55]:
def evaluate_parser(sentences):

    total_correct = 0
    total_words = 0

    for i, sentence in enumerate(sentences):

        # Parse the sentence
        parsed = parse_sentence(sentence)

        # Compare predicted arcs with gold arcs
        for token in sentence:

            token_id = token["id"]

            gold_head = token["head"]
            gold_relation = token["deprel"]

            total_words += 1

            # Skip if no prediction was produced
            if token_id not in parsed.arcs:
                continue

            predicted_head, predicted_relation = parsed.arcs[token_id]

            # LAS requires both head and relation to be correct
            if (
                predicted_head == gold_head
                and predicted_relation == gold_relation
            ):
                total_correct += 1

        # Show progress every 500 sentences
        if (i + 1) % 500 == 0:
            print("Processed", i + 1, "sentences")

    las = total_correct / total_words

    return las

In [54]:
dev_las = evaluate_parser(dev_sentences)

print(f"Development LAS: {dev_las * 100:.2f}%")

Processed 500 sentences
Processed 1000 sentences
Processed 1500 sentences
Processed 2000 sentences
Development LAS: 56.71%


In [56]:
test_las = evaluate_parser(test_sentences)

print(f"Test LAS: {test_las * 100:.2f}%")

Processed 500 sentences
Processed 1000 sentences
Processed 1500 sentences
Processed 2000 sentences
Test LAS: 57.17%


In [52]:
example_sentences = [
    
    [
        {"id": 1, "form": "The", "upos": "DET"},
        {"id": 2, "form": "cat", "upos": "NOUN"},
        {"id": 3, "form": "sat", "upos": "VERB"},
        {"id": 4, "form": "on", "upos": "ADP"},
        {"id": 5, "form": "the", "upos": "DET"},
        {"id": 6, "form": "mat", "upos": "NOUN"},
        {"id": 7, "form": ".", "upos": "PUNCT"}
    ],
    
    [
        {"id": 1, "form": "She", "upos": "PRON"},
        {"id": 2, "form": "eats", "upos": "VERB"},
        {"id": 3, "form": "a", "upos": "DET"},
        {"id": 4, "form": "green", "upos": "ADJ"},
        {"id": 5, "form": "salad", "upos": "NOUN"},
        {"id": 6, "form": ".", "upos": "PUNCT"}
    ],
    
    [
        {"id": 1, "form": "I", "upos": "PRON"},
        {"id": 2, "form": "saw", "upos": "VERB"},
        {"id": 3, "form": "the", "upos": "DET"},
        {"id": 4, "form": "man", "upos": "NOUN"},
        {"id": 5, "form": "with", "upos": "ADP"},
        {"id": 6, "form": "a", "upos": "DET"},
        {"id": 7, "form": "telescope", "upos": "NOUN"},
        {"id": 8, "form": ".", "upos": "PUNCT"}
    ]
]

print("Example sentences created successfully.")

Example sentences created successfully.


In [53]:
for i, example in enumerate(example_sentences, start=1):

    print("\n" + "=" * 50)
    print("Example Sentence", i)
    print("Sentence:", " ".join(token["form"] for token in example))
    print("=" * 50)

    parsed_example = parse_sentence(example)

    print("\nPredicted Dependencies:")

    for token in example:

        token_id = token["id"]
        word = token["form"]

        if token_id in parsed_example.arcs:

            head_id, relation = parsed_example.arcs[token_id]

            if head_id == 0:
                head_word = "ROOT"
            else:
                head_word = parsed_example.tokens[head_id]["form"]

            print(
                f"{word} -> {head_word} ({relation})"
            )


Example Sentence 1
Sentence: The cat sat on the mat .

Predicted Dependencies:
The -> cat (det)
cat -> ROOT (root)
sat -> cat (acl)
on -> mat (case)
the -> mat (det)
mat -> sat (obj)
. -> cat (punct)

Example Sentence 2
Sentence: She eats a green salad .

Predicted Dependencies:
She -> eats (nsubj)
eats -> ROOT (root)
a -> salad (det)
green -> salad (amod)
salad -> eats (obj)
. -> eats (punct)

Example Sentence 3
Sentence: I saw the man with a telescope .

Predicted Dependencies:
I -> saw (nsubj)
saw -> ROOT (root)
the -> man (det)
man -> saw (obj)
with -> telescope (case)
a -> telescope (det)
telescope -> man (nmod)
. -> saw (punct)


In [57]:
print("=" * 55)
print("DEPENDENCY PARSER - FINAL RESULTS")
print("=" * 55)

print("\nDataset:")
print("Training sentences:", len(train_sentences))
print("Development sentences:", len(dev_sentences))
print("Test sentences:", len(test_sentences))

print("\nTraining:")
print("Training examples:", len(X_train))
print("Failed oracle sentences:", 287)

print("\nEvaluation:")
print(f"Development LAS: {dev_las * 100:.2f}%")
print(f"Test LAS: {test_las * 100:.2f}%")

print("\nFeatures:")
print("1. POS of stack top")
print("2. POS of second stack item")
print("3. POS of first buffer item")
print("4. POS of second buffer item")

print("\nClassifier: Logistic Regression")

print("\nRequired Development LAS: {:.2f}%".format(dev_las * 100))
print("Additional Test LAS: {:.2f}%".format(test_las * 100))

print("=" * 55)

DEPENDENCY PARSER - FINAL RESULTS

Dataset:
Training sentences: 12544
Development sentences: 2001
Test sentences: 2077

Training:
Training examples: 407165
Failed oracle sentences: 287

Evaluation:
Development LAS: 56.71%
Test LAS: 57.17%

Features:
1. POS of stack top
2. POS of second stack item
3. POS of first buffer item
4. POS of second buffer item

Classifier: Logistic Regression

Required Development LAS: 56.71%
Additional Test LAS: 57.17%


## 9. Limitations and Observations

The implemented parser uses the Arc-Standard transition system with only four POS-based features. Therefore, the model has limited information about the lexical and structural context of a sentence, which affects parsing performance.

During oracle-based training-instance generation, 287 sentences could not be completely processed by the implemented oracle and were therefore skipped. The remaining sentences produced 407,165 valid training instances.

The development LAS was 56.71%, while the additional test evaluation achieved a LAS of 57.17%. The results demonstrate that the transition-based parser can learn dependency parsing decisions from oracle-generated training data using the specified POS-based feature representation.